# 02 — Stage 7 shared all-in precision repair validation

Each frozen original, mechanics and experimental predecessor receives the same exact sizing correction. The original r2 comparison remains unchanged before evidence. This run reuses its two authenticated preparation outputs and reruns the affected recommendation phases only. No models, cohorts, settings, gates or observed actions are changed. CPU High-RAM; run every code cell in order.

## 1. Mount and authenticate the frozen package
The local workspace is disposable; accepted phase receipts and outputs stay in this notebook-owned Drive runtime directory.

In [ ]:
import os, sys, json, hashlib, pathlib, subprocess, zipfile, tarfile, urllib.request, shutil, time, threading, uuid
assert sys.version_info[:2]==(3,12),sys.version
memory=dict(line.split(':',1) for line in pathlib.Path('/proc/meminfo').read_text().splitlines())
assert int(memory['MemTotal'].split()[0])*1024/1e9>=48,'CPU High-RAM is required before input preparation'
HEARTBEAT_STOP=threading.Event(); CURRENT_PHASE='authenticate_package'; NOTEBOOK_STARTED=time.monotonic()
def notebook_heartbeat():
    while not HEARTBEAT_STOP.wait(20): print({'phase':CURRENT_PHASE,'elapsed_seconds':round(time.monotonic()-NOTEBOOK_STARTED,1)},flush=True)
threading.Thread(target=notebook_heartbeat,daemon=True).start()
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS','VECLIB_MAXIMUM_THREADS','BLIS_NUM_THREADS'): os.environ[key]='1'
from google.colab import drive
if not pathlib.Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
DRIVE=pathlib.Path('/content/drive/MyDrive/Poker Assistant Model/Stage 7 - Empirical Assistant Layers')
PACKAGE=DRIVE/'inputs'/'stage7_frozen_capture_comparison_v2_r3.zip'
EXPECTED_PACKAGE={'bytes': 1150600983, 'sha256': '093a1b7e0342db58d06366d6f60f33f3b282364178387b224acb6545e591575f'}
def file_sha(p):
    with pathlib.Path(p).open('rb') as f: return hashlib.file_digest(f,'sha256').hexdigest()
assert PACKAGE.stat().st_size==EXPECTED_PACKAGE['bytes'] and file_sha(PACKAGE)==EXPECTED_PACKAGE['sha256']
WORK=pathlib.Path('/content/stage6_capture_'+EXPECTED_PACKAGE['sha256'][:16]); WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(PACKAGE) as z:
    names=z.namelist(); assert len(names)==len(set(names))
    manifest=json.loads(z.read('package_manifest.json'))
    assert names[-1]=='package_manifest.json' and set(names)==set(manifest['members'])|{'package_manifest.json'}
    for name,item in manifest['members'].items():
        p=pathlib.PurePosixPath(name); assert not p.is_absolute() and '..' not in p.parts and '\\' not in name and ':' not in name
        target=WORK/name; target.parent.mkdir(parents=True,exist_ok=True)
        if not target.exists():
            pending=target.with_name(target.name+'.partial-'+uuid.uuid4().hex)
            with z.open(name) as source, pending.open('xb') as dest: shutil.copyfileobj(source,dest,8*1024*1024)
            assert pending.stat().st_size==item['bytes'] and file_sha(pending)==item['sha256'],name
            assert not target.exists(); pending.rename(target)
        assert target.stat().st_size==item['bytes'] and file_sha(target)==item['sha256'],name
PROTOCOL=WORK/'protocol.json'; assert file_sha(PROTOCOL)=='844c80e50aceab3548d498555f0f9ccd663a9bbade76583d861acf59b2155407'
print({'authenticated_members':len(manifest['members']),'protocol_sha256':file_sha(PROTOCOL)})


## 2. Install exact Node dependencies and inspect CPU resources
The notebook defaults to at most two independent processes, one inner thread. Affinity and CPU quota determine usable capacity. Memory is recorded; it never reduces worker counts. If this is not the requested High RAM runtime, change the Colab runtime configuration before continuing.

In [ ]:
CURRENT_PHASE='install_pinned_node_dependencies'
sys.path.insert(0,str(WORK/'src'))
from stage6_empirical_capture_release_v2 import install_node_distribution
node=manifest['node_distribution']; NODE_ROOT=install_node_distribution(WORK,node)
os.environ['PATH']=str(NODE_ROOT/'bin')+os.pathsep+os.environ['PATH']
NODE=str(NODE_ROOT/'bin/node')
subprocess.run([NODE,str(NODE_ROOT/'lib/node_modules/npm/bin/npm-cli.js'),'ci','--ignore-scripts','--no-audit','--no-fund'],cwd=WORK,check=True)
subprocess.run([NODE,'--import','tsx','--input-type=module','-e',"import {transformSync} from 'esbuild'; if(!transformSync('const n: number=1',{loader:'ts'}).code)process.exit(1)"],cwd=WORK,check=True)
sys.path.insert(0,str(WORK/'src'))
from stage6_empirical_capture_v2 import cpu_capacity
memory=dict(line.split(':',1) for line in pathlib.Path('/proc/meminfo').read_text().splitlines())
print({'python':sys.version,'node':node['version'],'cpu_capacity':cpu_capacity(),'resolved_workers':min(2,cpu_capacity()),'total_ram_gb':int(memory['MemTotal'].split()[0])*1024/1e9,'ram_is_worker_gate':False})


## 3. Run causal preparation and three frozen comparisons
Completed phases resume only after their hashes match. Failed attempts remain preserved. Source and output hashes bind the run; no notebook edits can silently substitute another model or cohort.

In [ ]:
CURRENT_PHASE='execute_capture_comparison'
OUTPUT=DRIVE/'runtime_state'/('89_capture_v2_r1_'+'844c80e50aceab3548d498555f0f9ccd663a9bbade76583d861acf59b2155407'[:16])
process=subprocess.Popen([sys.executable,str(WORK/'src/stage6_empirical_capture_v2.py'),'--protocol',str(PROTOCOL),'--protocol-sha256','844c80e50aceab3548d498555f0f9ccd663a9bbade76583d861acf59b2155407','--package-root',str(WORK),'--output',str(OUTPUT),'--node',NODE,'--tsx-import',str(WORK/'node_modules/tsx/dist/loader.mjs')],cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout: print(line,end="",flush=True)
returncode=process.wait()
if returncode: raise subprocess.CalledProcessError(returncode,process.args)
report=json.loads((OUTPUT/'capture_comparison_report.json').read_text())
print({'disposition':report['disposition'],'roles':report['roles'],'qualification':report['qualification']})


## 4. Seal the direct result ZIP last
The receipt includes only accepted phase outputs, their receipts and the comparison report. The durable runtime remains available for exact resume. Mount readback is an internal check; external Drive download/readback and strict import remain required.

In [ ]:
CURRENT_PHASE='seal_result_archive'
from stage6_empirical_capture_release_v2 import seal_result
RESULT=DRIVE/'returned_results'/('89_capture_v2_r1_'+'844c80e50aceab3548d498555f0f9ccd663a9bbade76583d861acf59b2155407'[:16]+'.zip')
receipt=seal_result(OUTPUT,RESULT,EXPECTED_PACKAGE['sha256'],'844c80e50aceab3548d498555f0f9ccd663a9bbade76583d861acf59b2155407')
assert file_sha(RESULT)==receipt['sha256']
print({'result':str(RESULT),'bytes':RESULT.stat().st_size,'sha256':receipt['sha256'],'external_readback_and_import_required':True,'promotion_allowed':False})
CURRENT_PHASE='complete_pending_external_import'; HEARTBEAT_STOP.set()


## 5. Interpretation
Inspect exact action, size, gate and context differences by whole game and scenario. The prior three games remain previously inspected diagnostics. Calibrated range/continuation evidence and live UI/lifecycle verification are separate gates. A complete replay is not strategy qualification. Do not pool these hero-only captures with another cohort to claim its sample threshold.